Bibliotecas Necessárias

In [1]:
# Biblioteca que possibilita que o python manipule o sistema de arquivos
# do sistema operacional. Também vamos utilizar essa função para acessar
# as variáveis de ambiente que possuem as nossas informações de autenticação.
import os

# Biblioteca que tem como objetivo realizar requisições a sistemas e APIs
import requests

# Função do módulo auth da biblioteca requests que tem como objetivo realizar a
# autenticação do usuário em um sistema ou API. 
from requests.auth import HTTPBasicAuth

# Função da biblioteca dotenv que tem como objetivo carregar no código
# as variáveis de ambiente que serão acessadas pela biblioteca os.
from dotenv import load_dotenv

# Biblioteca que possibilita a manipulação de datas e horas
import datetime

# Biblioteca que possibilita a manipulação de dados para análise
import pandas as pd

Carregando variáveis de ambientes no código

In [2]:
# Função que ira carregar as variáveis de ambiente
load_dotenv()

# getenv: Função da biblioteca os que tem como objetivo acessar as variáveis de ambientes
# carregadas pelo load_dotenv. A função recebe como argumento o nome da variável que será
# acessada.
JIRA_URL = os.getenv("JIRA_URL")

EMAIL = os.getenv("EMAIL")

SENHA_API = os.getenv("SENHA_API")

Criando a pasta de logs de erros de automação

In [3]:
# makedirs: Função da biblioteca os que tem como objetivo criar diretórios no sistema
# arquivos do sistema operacional. A função recebe como argumento o caminho da criação
# do diretório e o exist_ok com um valor booleano que indica se o código deve ignorar
# ou não o trecho de criação de diretórios (é importante ignorar a criação caso o diretório
# ja exista). 
os.makedirs("Erros", exist_ok=True)

# Trecho que irá acessar e formatar o a data e a hora capturada no sistema, vamos usar essa variável
# para atribuir horário e data no nome dos arquivos com o objetivo de criar diferentes arquivos (com base
# em sua data e hora da criação) de erros. 
agora = datetime.datetime.now().strftime("%d_%m_%Y_%H_%M")

# Caminho completo que iremos utilizar na criação de arquivos de logs
caminho_erros = f"Erros/erros_{agora}.txt"

Conexão com  a API do Jira que contém as informações dos chamados

In [4]:
# autenticação: Variável que irá conter os valores necessários para realizarmos
# a nossa autenticação no jira. 

# HTTPBasicAuth: Função que tem como objetivo realizar a nossa autenticação na API
# utilizando as nossas credenciais. A função recebe como argumento o email da nossa
# conta no jira e a senha da nossa API  
autenticacao = HTTPBasicAuth(EMAIL, SENHA_API)

# cabecalho (header): Dicionário que tem como objetivo definir o escopo (regras) que irão ditar como
# o acesso aos dados devem acontecer.
cabecalho = {

    # Define que os parametros serão passados em formato de um dicionário json
    "Accept": "application/json",

    # Define que os dados retornados devem ser no formato json.
    "Content-Type": "application/json"
}

# Url de acesso a API que contém informações da nossa conta(JIRA_URL), das regras de acesso (rest),
# da versão da API (3), da ação que iremos realizar (search ou pesquisa em português) e a linguagem de
# consulta dos dados.
url = f"{JIRA_URL}/rest/api/3/search/jql"

# Dicionário que ira conter os dados que queremos acessar e como queremos acessar.
parametros = {

    # Consulta dos dados ordenados em ordem decrescente
    "jql":"project = Suporte ORDER BY created DESC",

    # Ira indicar a primeira linha que deve ser acessada
    "startAt": 0,

    # Ira dar um limite de linhas que devem ser retornadas a cada acesso a API
    "maxResults": 100,

    # Ira conter os campos que devem ser acessados pela API 
    "fields": ["summary", "assignee","reporter","priority","status", "resolution","created",  "updated"]
    
    }


# Ira inspecionar o bloco de conexão com o objetivo de capturar possiveis erros.
try:

    # get: Função da biblioteca requests que realiza requisições a API e sites.
    # Ela ira receber como argumento todas as informações necessárias para a
    # realização da nossa consulta na API. A função ira retornar um objeto
    # que contém as informações retornadas pelo servidor (tanto positivas
    # quanto negativas).

    # url: Contém o caminho de acesso a API.
    # headers: Recebe o cabeçalho de regras de acesso aos dados
    # auth: Reecebe os dados da nossa credencial
    # params: Contém o dicionário com as informações que queremos acessar
    resposta = requests.get(
    url,
    headers = cabecalho,
    auth = autenticacao,
    params = parametros)

    # Estrutura condicional que ira verificar o codigo retornado pelo servidor
    # da API

    # Ira verificar se o código retornado é do tipo 200 (acesso bem sucedido)
    # status_code: Atributo do objeto retornado pelo get que contém o código
    # retornado pelo servidor após uma requisição.
    if resposta.status_code == 200:

        # Se essa condição for verdadeira, vamos imprimir no terminal essa
        # mensagem.
        print("Conexão bem sucedida com a API")
    
    else:

        # Caso o servidor retorne outro tipo de código, vamos iniciar o processo
        # de registro do ocorrido no arquivo de logs de erros.

        # Impressão de uma mensagem utilizando o código retornado pelo servidor.
        print(f"Falha na conexao. código retornado: {resposta.status_code}")

        # with: Garante que os arquivo sejam fechados/encerrados após a execução
        # do código com o objetivo de evitar que o arquivo fique corrompido.

        # open: Funçãoque tem como objetivo criar ou abrir arquivos ja existentes
        # no projeto. Ela recebe como argumento o caminho do arquivo que esta sendo
        # criado ou aberto, o comando que será realizado para manipula-lo e a
        # codificação que indica como o python deve interpretar os dados.

        # caminhos_erros: Caminho de criação dos arquivos

        # a: Como append que ira adicionar valores nos arquivos criados. Observação:
        # se utilizarmos o comando 'w' (write) o python sempre irá sobrescrever
        # os dados registrados na execução anterior.

        # encoding="utf-8": Codificação que suporta a utilização de caracteres especiais

        # erro: Apelido que irá representar o arquivo que iremos criar e manipular.

        # write: Função que irá registrar dados no arquivo criado.

        # text: Atributo do objeto retornado pelo get que contém o texto
        # que indica a divergência encontrada.

        # agora: Ira conter a data e a hora da requisição a API.

        # status_code: Atributo que contém o codigo retornado pelo servidor
        with open(f"{caminho_erros}", "a", encoding="utf-8") as erros:

            erros.write(f"Falha na conexão, código retornado: {resposta.status_code}\n")

            erros.write(f"Motivo: {resposta.text}\n")

            agora = datetime.datetime.now().strftime("%d_%m_%Y_%H_%M")

            erros.write(f"Data e hora da Requisição: {agora}\n")

            erros.write("------------------------------------------------\n")
            
except requests.exceptions.RequestException as erro:

    # Exceção que ira lidar com erros na requisição a API. Ela ira repetir
    # todo o processo que documentamos anteriormente
    print(f"Falha na conexão com a API: {erro}")

    with open(f"{caminho_erros}", "a", encoding="utf-8") as erros:

            erros.write(f"Falha na conexão, código retornado: {erro}\n")

            erros.write(f"Motivo: {resposta.text}\n")

            agora = datetime.datetime.now().strftime("%d_%m_%Y_%H_%M")

            erros.write(f"Data e hora da Requisição: {agora}\n")

            erros.write("------------------------------------------------\n")


Conexão bem sucedida com a API


In [12]:
dados_api = resposta.json()

issues = dados_api.get("issues")

print(issues)

[{'expand': 'renderedFields,names,schema,operations,editmeta,changelog,versionedRepresentations', 'id': '10017', 'self': 'https://caikedom.atlassian.net/rest/api/3/issue/10017', 'key': 'SUP-18', 'fields': {'summary': 'queda da funcionalidade de login', 'created': '2026-10-07T12:23:01.588-0300', 'reporter': {'self': 'https://caikedom.atlassian.net/rest/api/3/user?accountId=5f8b9fe114dd460068eb8e16', 'accountId': '5f8b9fe114dd460068eb8e16', 'emailAddress': 'caike.dom@gmail.com', 'avatarUrls': {'48x48': 'https://secure.gravatar.com/avatar/2537a0922389dc5b6675be8037e54d9f?d=https%3A%2F%2Favatar-management--avatars.us-west-2.prod.public.atl-paas.net%2Finitials%2FCD-5.png', '24x24': 'https://secure.gravatar.com/avatar/2537a0922389dc5b6675be8037e54d9f?d=https%3A%2F%2Favatar-management--avatars.us-west-2.prod.public.atl-paas.net%2Finitials%2FCD-5.png', '16x16': 'https://secure.gravatar.com/avatar/2537a0922389dc5b6675be8037e54d9f?d=https%3A%2F%2Favatar-management--avatars.us-west-2.prod.public.

    "fields": ["summary", "assignee","reporter","priority","status", "resolution","created",  "updated","issuetype",  ]


In [6]:
for issue in issues:

    fields = issue.get("fields")

    for chave, campo in fields.items():

        print(f"Chave:{chave} | Tipo do dado: {type(campo)}")

Chave:summary | Tipo do dado: <class 'str'>
Chave:created | Tipo do dado: <class 'str'>
Chave:reporter | Tipo do dado: <class 'dict'>
Chave:assignee | Tipo do dado: <class 'NoneType'>
Chave:priority | Tipo do dado: <class 'dict'>
Chave:resolution | Tipo do dado: <class 'NoneType'>
Chave:updated | Tipo do dado: <class 'str'>
Chave:status | Tipo do dado: <class 'dict'>
Chave:summary | Tipo do dado: <class 'str'>
Chave:created | Tipo do dado: <class 'str'>
Chave:reporter | Tipo do dado: <class 'dict'>
Chave:assignee | Tipo do dado: <class 'NoneType'>
Chave:priority | Tipo do dado: <class 'dict'>
Chave:resolution | Tipo do dado: <class 'NoneType'>
Chave:updated | Tipo do dado: <class 'str'>
Chave:status | Tipo do dado: <class 'dict'>
Chave:summary | Tipo do dado: <class 'str'>
Chave:created | Tipo do dado: <class 'str'>
Chave:reporter | Tipo do dado: <class 'dict'>
Chave:assignee | Tipo do dado: <class 'NoneType'>
Chave:priority | Tipo do dado: <class 'dict'>
Chave:resolution | Tipo do dad

        Tipos de cada dado do dicionário field

summary: str

assignee: NoneType

reporter: dict

priority: dict

status: dict

resolution: NoneType

created: str

uptaded: str

issuetype: dict

In [ ]:
lista_campos = []

for issue in issues:

    fields = issue.get("fields")

    summary = fields.get("summary")

    assignee = fields.get("assignee")

    reporter = fields.get("displayName")

    priority = fields.get("displayName")

    status = fields.get("displayName")

    resolution = fields.get("resolution")

    created = fields.get("created")

    uptaded = fields.get("updated")

    issuetype = fields.get("displayName")

    lista_campos.append({"ticket": summary,
                          "responsável": assignee,
                          "relator": reporter,
                          "prioridade": priority,
                          "status": status,
                          "resolucao": resolution,
                          "criado": created,
                          "atualizacao": uptaded,
                          "tipo_issue": issuetype
    })


In [8]:
lista_campos

[{'ticket': 'queda da funcionalidade de login',
  'responsável': None,
  'relator': None,
  'prioridade': None,
  'status': None,
  'resolucao': None,
  'criado': '2026-10-07T12:23:01.588-0300',
  'atualizacao': '2026-10-07T12:23:02.389-0300',
  'tipo_issue': None},
 {'ticket': 'Atualização da página de sobre',
  'responsável': None,
  'relator': None,
  'prioridade': None,
  'status': None,
  'resolucao': None,
  'criado': '2026-10-07T12:10:36.365-0300',
  'atualizacao': '2026-10-07T12:10:37.159-0300',
  'tipo_issue': None},
 {'ticket': 'Falha nas atualizações de cadastro dos clientes',
  'responsável': None,
  'relator': None,
  'prioridade': None,
  'status': None,
  'resolucao': None,
  'criado': '2026-10-07T11:53:36.240-0300',
  'atualizacao': '2026-10-07T11:53:36.955-0300',
  'tipo_issue': None},
 {'ticket': 'Erro no carregamento da página de produtos',
  'responsável': None,
  'relator': None,
  'prioridade': None,
  'status': None,
  'resolucao': {'self': 'https://caikedom.atla

In [9]:
df = pd.DataFrame(lista_campos)

In [10]:
df.head(18)

,ticket,responsável,relator,prioridade,status,resolucao,criado,atualizacao,tipo_issue
0,queda da funcionalidade de login,None,None,None,None,None,2026-10-07T12:23:01.588-0300,2026-10-07T12:23:02.389-0300,None
1,Atualização da página de sobre,None,None,None,None,None,2026-10-07T12:10:36.365-0300,2026-10-07T12:10:37.159-0300,None
2,Falha nas atualizações de cadastro dos clientes,None,None,None,None,None,2026-10-07T11:53:36.240-0300,2026-10-07T11:53:36.955-0300,None
3,Erro no carregamento da página de produtos,None,None,None,None,{'self': 'https://caikedom.atlassian.net/rest/...,2026-10-07T11:41:18.653-0300,2026-10-07T12:26:24.884-0300,None
4,Atualização do front-end,None,None,None,None,None,2026-10-06T17:27:11.997-0300,2026-10-06T17:27:12.657-0300,None
5,Falha no backup dos dados,None,None,None,None,None,2026-10-06T17:06:02.051-0300,2026-10-06T17:14:37.106-0300,None
6,problemas no botão de envio na tela de cadastro,None,None,None,None,None,2026-10-06T17:02:42.997-0300,2026-10-06T17:19:41.513-0300,None
7,Atualização de servidores,None,None,None,None,None,2026-10-06T16:58:13.325-0300,2026-10-06T16:58:14.079-0300,None
8,Falha no cadastro de novos clientes,None,None,None,None,None,2026-10-06T16:50:06.333-0300,2026-10-06T16:50:07.185-0300,None
9,Queda do Sistema,{'self': 'https://caikedom.atlassian.net/rest/...,None,None,None,None,2026-10-06T16:44:55.765-0300,2026-10-07T12:27:36.099-0300,None
